In [9]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from core.config import load_config, print_config
from core.data.loader import (
    setup_database, load_and_group_data_from_db, 
    calculate_normalization_stats, create_sampler_from_config, print_data_summary
)
from core.data.dataset import create_dataloaders
from core.models.factory import create_task_model_from_config, print_model_info
from core.training.trainer import setup_training
from core.data.types import HitObjectVector, BeatmapMetadata, VECTOR_DIM, METADATA_DIM

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

print(f"Vector fields: {HitObjectVector.get_field_names()}")
print(f"Metadata fields: {BeatmapMetadata.get_field_names()}")

PyTorch version: 2.8.0+cu126
Using device: cuda
Working directory: /home/jessiez/osu_corpora
Vector fields: ['distance_diff', 'angle_cos', 'angle_sin', 'time_diff', 'abs_x', 'abs_y', 'is_circle', 'is_slider', 'is_spinner', 'is_new_combo', 'slider_curve_b', 'slider_curve_c', 'slider_curve_l', 'slider_curve_p', 'slider_num_anchors', 'slider_pixel_length', 'duration_beats']
Metadata fields: ['ar', 'od', 'cs', 'difficulty_rating', 'bpm']


In [10]:
CONFIG_NAME = "bert" 

config = load_config(CONFIG_NAME, config_dir="configs")
print_config(config, f"Loaded Configuration: {CONFIG_NAME}")


--- Loaded Configuration: bert ---
data:
  db_path: ./data/beatmaps_test.db
  max_seq_len: 1023
  in_channels: 17
  val_split: 0.1
  chunk_size: 1000
training:
  batch_size: 8
  num_epochs: 5
  learning_rate: 0.0005
  weight_decay: 0.05
  warmup_ratio: 0.05
  min_lr: 1.0e-06
  use_amp: true
  checkpoint_dir: ./checkpoints/bert
  grad_clip_norm: 1.0
  sampling:
    method: temperature
    kde_bandwidth: 0.3
    kde_bins: 200
    temperature: 2.0
    difficulty_index: 3
    expand_for_augmentation: false
model:
  dropout: 0.1
  type: bert
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
components:
  use_rope: true
  use_flash_attention: true
  compile_model: true
mlm:
  masking_ratio: 0.2

----------------------------------


In [11]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
db_path = setup_database(config['data']['db_path'], colab_url)

print(f"Using database: {db_path}")

all_beatmaps_data = load_and_group_data_from_db(
    db_path, 
    chunk_size=config['data'].get('chunk_size', 1000),
    max_seq_len=config['data']['max_seq_len']
)

print_data_summary(all_beatmaps_data)

Using database: ./data/beatmaps_test.db
Connecting to database...
Fetching valid beatmap IDs and all metadata...
Found 2833 beatmaps with complete metadata.
Processing Chunks: 100%|██████████| 3/3 [00:09<00:00,  3.01s/it]
Finished processing all data.

--- Data Summary ---
Total beatmaps: 2833
Vector dimension: 17
Metadata dimension: 5
Sequence length - Min: 35, Max: 1023, Avg: 701.5
--------------------


In [12]:
from torch.utils.data import random_split

val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size
train_data, val_data = random_split(all_beatmaps_data, [train_size, val_size])

print(f"Data split: {len(train_data)} training, {len(val_data)} validation")

train_data_list = [train_data.dataset[i] for i in train_data.indices]
val_data_list = [val_data.dataset[i] for i in val_data.indices]

sampler = create_sampler_from_config(train_data_list, config)

normalizer = calculate_normalization_stats(
    train_data_list,
    include_augmentation=config['training']['sampling']['expand_for_augmentation']
)

vector_mean, vector_std = normalizer.get_vector_stats()
meta_mean, meta_std = normalizer.get_metadata_stats()

Data split: 2550 training, 283 validation
Creating temperature sampler with temperature=2.0...
Temperature sampling - Min weight: 0.6365, Max weight: 7.5166
Calculating normalization statistics...

                    NORMALIZATION STATISTICS

--- VECTOR STATISTICS:
----------------------------------------------------------------------
Field Name           Mean         Std Dev      Normalized  Description
----------------------------------------------------------------------
distance_diff        4.2971       1.5371       Yes         Distance between objects (log)
angle_cos            0.0844       0.7464       Yes         Angle cosine component
angle_sin            0.0048       0.6601       Yes         Angle sine component
time_diff            0.3364       0.1641       Yes         Time difference (log)
abs_x                -0.0045      0.5191       Yes         Absolute X position (scaled)
abs_y                0.0112       0.5397       Yes         Absolute Y position (scaled)
is_circle  

In [13]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['training']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}, meta={sample_batch[2].shape}")

Created dataloaders with batch size: 8
Sample batch shapes: vectors=torch.Size([8, 1023, 17]), mask=torch.Size([8, 1023]), meta=torch.Size([8, 5])


In [ ]:
torch.set_float32_matmul_precision('high')
model = create_task_model_from_config(config, device, task_type='mlm')

print_model_info(model, config)

with torch.no_grad():
    sample_vectors, sample_mask, sample_metadata = sample_batch
    predictions, targets, _ = model(sample_vectors, sample_metadata, sample_mask)
    print(f"Model test - Predictions: {predictions.shape}, Targets: {targets.shape}")
    
print("\n✅ Model created and tested successfully!")

Compiling task model with torch.compile...

--- Model Information ---
Model Type: bert
Total Parameters: 25.46M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6
RoPE Enabled: True
Flash Attention: True
Model Compiled: True
-------------------------


ValueError: too many values to unpack (expected 2)

In [ ]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        start_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch += 1 
        print(f"Loaded checkpoint, resuming from epoch {start_epoch + 1}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting training from scratch")

print(f"Training setup complete. Starting from epoch {start_epoch + 1}")
print(f"Total epochs: {config['training']['num_epochs']}")

In [ ]:

print("\n🚀 Starting training...")
print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {config['model']['type']} with {config['model']['n_layers']} layers")
print(f"Components: RoPE={config['components'].get('use_rope', False)}")

if config.get('training', {}).get('sampling', {}).get('expand_for_augmentation', True):
    effective_train_size = len(train_data) * 4 if config.get('training', {}).get('sampling', {}).get('expand_for_augmentation', True) else len(train_data)
    print(f"Training samples: {len(train_data)} base maps -> {effective_train_size} with augmentation")
else:
    print(f"Training samples: {len(train_data)} base maps without augmentation")

metrics_tracker = trainer.train(start_epoch)

print("\n🎉 Training completed!")
print("Training history saved in metrics_tracker")